# HDB resale data: transformation and de-identification

This notebook continues from `01_eda.ipynb`. It reads the cleaned January 2012–December 2017 dataset, creates unique resale identifiers, produces the de-identified output, and regenerates the final profiling report. Run the cells from top to bottom. The private HMAC pepper must be supplied through `HDB_IDENTIFIER_PEPPER`; it is never printed or saved in the notebook.

In [1]:
from pathlib import Path
import sys

import pandas as pd

candidates = [Path.cwd(), Path.cwd() / "part1_data_pipeline", Path.cwd().parent]
part1 = next(path for path in candidates if (path / "data" / "cleaned").is_dir())
sys.path.insert(0, str(part1.resolve()))

cleaned_path = part1 / "data" / "cleaned" / "resale_cleaned_2012_2017.csv"
if not cleaned_path.exists():
    raise FileNotFoundError("Run 01_eda.ipynb first to create the cleaned dataset")
cleaned = pd.read_csv(cleaned_path, dtype="string", keep_default_na=False)
print(f"Cleaned input records: {len(cleaned):,}")

Cleaned input records: 105,225


## 1. Create unique resale identifiers

Build the readable base identifier from block digits, the average resale price for each month/town/flat-type group, `YYMM`, and the town initial. When a base value repeats, order records by source file and source record number. The first keeps the base value; later records receive `-002`, `-003`, and so on. This retains every accepted record and is reproducible for the same input snapshot.

In [2]:
from transform import build_transformed

transformed_path = part1 / "data" / "transformed" / "resale_transformed_2012_2017.csv"
transformed = build_transformed(cleaned_path, transformed_path)
collision_rows = transformed["resale_identifier_base"].duplicated(keep=False)
print(f"Transformed records: {len(transformed):,}")
print(f"Records in colliding groups: {collision_rows.sum():,}")
print(f"Colliding base identifiers: {transformed.loc[collision_rows, 'resale_identifier_base'].nunique():,}")
print(f"Records receiving a suffix: {transformed['resale_identifier'].ne(transformed['resale_identifier_base']).sum():,}")
print(f"Final identifiers unique: {transformed['resale_identifier'].is_unique}")
display(transformed.loc[collision_rows,
                        ["source_file", "source_record_number", "resale_identifier_base",
                         "resale_identifier"]].head())

Transformed records: 105,225
Records in colliding groups: 24,406
Colliding base identifiers: 11,011
Records receiving a suffix: 13,395
Final identifiers unique: True


,source_file,source_record_number,resale_identifier_base,resale_identifier
1,ResaleFlatPricesBasedonApprovalDate2000Feb2012...,366465,S3142561201A,S3142561201A
2,ResaleFlatPricesBasedonApprovalDate2000Feb2012...,366466,S3142561201A,S3142561201A-002
15,ResaleFlatPricesBasedonApprovalDate2000Feb2012...,366479,S5583471201A,S5583471201A
21,ResaleFlatPricesBasedonApprovalDate2000Feb2012...,366485,S5503471201A,S5503471201A
22,ResaleFlatPricesBasedonApprovalDate2000Feb2012...,366486,S5503471201A,S5503471201A-002


## 2. De-identify resale identifiers

Apply HMAC-SHA-256 with the private `HDB_IDENTIFIER_PEPPER` environment variable. The secret key prevents straightforward dictionary attacks against the predictable identifier format. The output removes both readable identifier columns and retains the cleaned fields.

In [3]:
from hash_identifiers import build_hashed

hashed_path = part1 / "data" / "hashed" / "resale_hashed_2012_2017.csv"
hashed = build_hashed(transformed_path, hashed_path)
print(f"Hashed records: {len(hashed):,}")
print(f"Hashed identifiers unique: {hashed['hashed_resale_identifier'].is_unique}")
print(f"Readable identifiers present: {bool({'resale_identifier', 'resale_identifier_base'} & set(hashed.columns))}")

Hashed records: 105,225
Hashed identifiers unique: True
Readable identifiers present: False


## 3. Generate the final profiling report

Regenerate the report after all five output groups exist. The report reconciles lineage and row counts, verifies the transformation, and checks hashed identifier format and uniqueness without requiring access to the private pepper.

In [4]:
import subprocess

report_path = part1 / "DATA_PROFILING_REPORT.md"
subprocess.run([sys.executable, str(part1 / "profile.py")],
               check=True, capture_output=True, text=True)
print(f"Created {report_path.name} ({len(report_path.read_text().splitlines())} lines)")

Created DATA_PROFILING_REPORT.md (288 lines)


## Results and assumptions

- The required components produce **11,011 colliding base identifiers** involving **24,406 records**.
- Deterministic source-lineage suffixes are added to **13,395 records**, retaining all rows and making all **105,225 final identifiers unique**.
- HMAC-SHA-256 produces **105,225 unique de-identified identifiers**. Both readable identifier columns are excluded from the hashed output.
- The HMAC pepper must contain at least 32 characters and remain outside source control. Hashes generated with a different pepper will differ by design.